# Descriptive Analysis

Dengue federated analysis workshop.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ISARICResearch/Dissemination/blob/main/workshops/federated%20analytics/FederatedAnalysis_Project/notebooks/02_descriptive_analysis.ipynb)

## Introduction

Descriptive analysis summarizes the main characteristics of a dataset through a small set of statistics. For continuous variables, such as age, it uses measures like the mean, standard deviation, median and quartiles, while for categorical variables, such as sex, it counts how many records fall into each category. It gives a clear picture of the population being studied, and is usually the first result reported in a clinical or epidemiological study.

This notebook walks through the full process of calculating and comparing descriptive statistics using the workshop's dummy dengue dataset.

## Objective

Compare the descriptive statistics obtained with the centralized and federated approaches, for both continuous and categorical variables, to evaluate whether they match and identify where they diverge. The workshop uses a dummy dengue dataset from 5 health institutions.

## Table of Contents

```
1. Setup
    1.1. Environment detection
    1.2. Install dependencies
    1.3. Get project files
    1.4. Imports
    1.5. Load data
2. Data Preparation
    2.1. Drop unused columns
3. Descriptive Statistics Function
4. Centralized Approach
5. Federated Approach
    5.1. Per-site statistics
    5.2. Combine site results
    5.3. Aggregated calculations
6. Comparison
7. Conclusions
```

## 1. Setup

Prepare everything the notebook needs before starting the analysis.

### 1.1. Environment detection

Check whether this notebook is running on Google Colab or on a local computer.

In [14]:
import sys

# True on Google Colab, False on a local computer
IN_COLAB = "google.colab" in sys.modules

### 1.2. Install dependencies

Install any extra packages this notebook needs. This step only runs on Google Colab. On a local computer, the packages are already installed.

In [15]:
# if IN_COLAB:
#     %pip install -q <package>

### 1.3. Get project files

Get the project's data and code. On Google Colab, only the project's folder is downloaded first from the repository where it's published; on a local computer, they're already available next to this notebook.

In [16]:
from pathlib import Path

# Repository the project is downloaded from on Google Colab, and the project's folder inside it
REPO_URL, PROJECT_PATH = "https://github.com/ISARICResearch/Dissemination.git", "workshops/federated analytics/FederatedAnalysis_Project"
REPO_NAME = Path(REPO_URL).stem

if IN_COLAB:
    # Download only this project's folder, not the whole repository
    project_dir = Path(REPO_NAME) / PROJECT_PATH
    if not project_dir.exists():
        !git clone --depth 1 --filter=blob:none --sparse {REPO_URL}
        !git -C {REPO_NAME} sparse-checkout set "{PROJECT_PATH}"
else:
    # This notebook already lives inside the project
    project_dir = Path("..")

# Make the project's own code importable (e.g. `analysis_lib`)
sys.path.insert(0, str(project_dir))

data_dir = project_dir / "data"
data_file = data_dir / "dummy_data.parquet"
dictionary_file = data_dir / "dummy_data_dictionary.json"

### 1.4. Imports

Load the Python libraries used in this notebook.

In [17]:
# Standard library
import json

# Third-party
import pandas as pd
from IPython.display import Markdown, display

# Project
from analysis_lib.preprocessing import drop_columns_by_category
from analysis_lib.descriptive_analysis import descript_stats, result_df, weighted_avg, weighted_std, process_comparison

### 1.5. Load data

Read the example data.

In [18]:
# Load the full example dataset
df = pd.read_parquet(data_file)

# Load the data dictionary that describes each column
with open(dictionary_file, encoding="utf-8") as f:
    data_dictionary = json.load(f)

## 2. Data Preparation

Prepare the dataset for the descriptive analysis.

### 2.1. Drop unused columns

Remove the columns not needed for the descriptive analysis: identification (`record_id`) and dates.

- `record_id` is unique for each record, so it doesn't describe any characteristic of the patients.
- Dates show when each event happened (consultation, symptom onset, hospitalization and death), not a characteristic of the population: what they're useful for is the time between events, used later in the survival analysis.

In [19]:
# Drop identification and date columns, not used in the descriptive analysis
df = drop_columns_by_category(df, data_dictionary, categories_to_drop=["Identification", "Date"])

## 3. Descriptive Statistics Function

Define the function that computes descriptive statistics for a set of records. This is the common procedure both the centralized and federated approaches use:

- Centralized approach: calls it once over the whole dataset.
- Federated approach: calls the exact same function once per institution, over that institution's own data only.

It reports the following measures for each variable:

| Measure | Variables | What it shows |
|---|---|---|
| `mean (sd)` | Continuous | Average value, and how far values typically lie from it (standard deviation). |
| `median` | Continuous | Value with half of the records below it and half above. Less affected by extreme values than the mean. |
| `q1 - q3` | Continuous | Values between which the central 50% of the data lies (interquartile range). |
| `min-max` | Continuous | Full range of observed values. |
| Age groups (`0-9` to `80+`) | `age` | Count (%) of patients in each 10-year range. |
| Each category | Categorical | Count (%) of records in each category. |
| `missing` | All | Records with no value. |
| `n_data` | All | Records with a value, used as the base for the percentages. |

In this dataset, `age` is the only continuous variable; all the others are categorical.

In [20]:
def compute_descriptive_stats(df: pd.DataFrame, name: str) -> dict[str, pd.DataFrame]:
    """Compute each column's descriptive statistics.

    Args:
        df: Dataset to summarize.
        name: Label identifying this data (e.g. "Centralized" or a site name).

    Returns:
        Mapping of column name to a one-row DataFrame with its statistics.
    """
    return {
        column: result_df([descript_stats(df[column], name=name)])
        for column in df.columns
    }

## 4. Centralized Approach

Compute descriptive statistics over the whole dataset, as if all institutions shared their data directly.

In [21]:
# Apply the common function to the whole dataset
central_results = compute_descriptive_stats(df, name="Centralized")

Show the result for three examples: a continuous variable (`age`) and two categorical ones (`outcome`, and `shock`, which has missing values).

In [22]:
# Show one continuous and two categorical variables as examples
for variable in ["age", "outcome", "shock"]:
    is_numeric = pd.api.types.is_numeric_dtype(df[variable])
    # sum/sum_squares are only needed for the federated calculation in 5.3, not for display
    central_table = central_results[variable].drop(columns=["sum", "sum_squares"], errors="ignore")

    display(Markdown(f"#### {variable}"))
    display(process_comparison(central_table, is_numeric, variable))

#### age

Characteristic,Centralized
mean (sd),31.08 (19.44)
median,25.0
q1 - q3,15.0 - 43.0
min-max,7 - 91
missing,0
n_data,10284
0-9,440 (4.28%)
10-19,3541 (34.43%)
20-29,1977 (19.22%)
30-39,1334 (12.97%)


#### outcome

Characteristic,Centralized
missing,0
n_data,10284
Alive,10186 (99.05%)
Dead,98 (0.95%)


#### shock

Characteristic,Centralized
missing,1687
n_data,8597
No,8369 (97.35%)
Yes,228 (2.65%)


## 5. Federated Approach

Compute descriptive statistics per institution, then combine them without sharing individual records. Same metrics as in section 3, computed per site.

### 5.1. Per-site statistics

For each institution, apply the same calculations as the centralized approach, but only within that institution's own data. Each institution only shares its results, never its raw individual records.

The following code goes through each institution, simulating the calculations it would run locally and independently, without sharing raw records with anyone. Each institution's result is stored in the `local_results` dictionary, keyed by institution, as if this were what a central server would receive from every site under a real federated setup.

In [23]:
# Simulate each institution computing its own statistics independently,
# using the exact same function as the centralized approach
local_results = {}
for site in sorted(df["institution"].unique()):
    site_df = df.loc[df["institution"] == site]
    local_results[site] = compute_descriptive_stats(site_df, name=site)

### 5.2. Combine site results

Join each institution's local results into a single table per variable, one row per site.

Once each institution's results are available in `local_results`, they need to be reorganized by variable instead of by site, so every site's row for a given variable is available together.

This is only a reshaping step over the results each site already shared, no individual records are involved.

The resulting `sites_results` also lets you compare how each site's local statistics differ, before combining them into a single aggregated result per variable in the next step.

In [24]:
# Combine each site's local results into a single table per column, one row per site
sites_results = {}
for column in df.columns:
    site_tables = [local_results[site][column] for site in local_results]
    sites_results[column] = pd.concat(site_tables)

### 5.3. Aggregated calculations

Combine each variable's per-site table into a single aggregated result, analogous to `central_results`, so both approaches can be compared side by side in section 6.

How each metric is combined depends on what it actually represents, since not every centralized calculation can be exactly reproduced from separate per-site summaries.

`1)` `min` and `max` are exact: the overall minimum/maximum is simply the minimum/maximum across all sites' local values.

`2)` `missing`, `n_data`, and the category/age-bin counts are exact as well, since they are plain counts and can be summed directly across sites.

`3)` `mean` is also exact: statistically, a weighted average of each site's mean, weighted by its number of records with a value ($n_i$, `n_data`), equals the mean of the pooled data. This is the same weighted-averaging principle behind `weighted_avg`:

$$\bar{x} = \frac{\sum_{i=1}^{k} n_i \bar{x}_i}{\sum_{i=1}^{k} n_i}$$

where $k$ is the number of sites, $n_i$ is site $i$'s number of records with a value, and $\bar{x}_i$ is site $i$'s local value for the metric.

`4)` `sd` requires a dedicated formula (`weighted_std`), built from each site's sum and sum of squares, the two additional values shared for this purpose. This is the standard computational ("naive") formula for variance, applied to the sums pooled across all sites:

$$s^2 = \frac{\sum x_j^2}{n - 1} - \frac{\left(\sum x_j\right)^2}{n(n-1)}, \qquad s = \sqrt{s^2}$$

where $n = \sum_{i=1}^{k} n_i$, and $\sum x_j$ and $\sum x_j^2$ are each site's sum and sum of squares, added together across sites. Since sums are additive, this reconstructs the exact standard deviation of the pooled data, the same way the centralized approach would compute it directly.

`5)` `median`, `q1` and `q3` cannot be reconstructed exactly from per-site summaries alone: unlike sums or counts, these are order statistics that depend on the position of individual values, not on aggregatable quantities. `weighted_avg` (the same formula shown above for `mean`) is used here too, as an approximation, but it will not necessarily match the centralized result exactly, as shown in section 6.

In [25]:
federated_results = {}
for variable in sites_results:
    df_variable = sites_results[variable]
    results_metric = {}

    # Metric names to combine, excluding sum/sum_squares (internal, only needed for weighted_std)
    metrics = [metric for metric in df_variable.columns if metric not in ("sum", "sum_squares")]
    for metric in metrics:
        if metric == "min":
            # Exact: the overall minimum is the minimum across all sites
            results_metric[metric] = df_variable[metric].min()
        elif metric == "max":
            # Exact: the overall maximum is the maximum across all sites
            results_metric[metric] = df_variable[metric].max()
        elif metric in ("mean", "median", "q1", "q3"):
            # Exact for mean; an approximation for median, q1 and q3
            results_metric[metric] = weighted_avg(df_variable[[metric, "n_data"]], metric)
        elif metric == "sd":
            # Exact: reconstructed from each site's sum and sum of squares
            results_metric[metric] = weighted_std(df_variable[["n_data", "sum", "sum_squares"]])
        else:
            # missing, n_data and category/age-bin counts: exact, simple sum across sites
            results_metric[metric] = df_variable[metric].astype(float).sum()

    results_metric["name"] = "Federated"
    federated_results[variable] = result_df([results_metric])

## 6. Comparison

Compare the centralized and federated results to check whether they match.

In [26]:
for variable in df.columns:
    is_numeric = pd.api.types.is_numeric_dtype(df[variable])

    # sum/sum_squares are only needed for the federated calculation in 5.3, not for display
    central_table = central_results[variable].drop(columns=["sum", "sum_squares"], errors="ignore")
    site_table = sites_results[variable].drop(columns=["sum", "sum_squares"], errors="ignore")
    federated_table = federated_results[variable]

    df_total = pd.concat([central_table, federated_table, site_table])
    df_result = process_comparison(df_total, is_numeric, variable)

    display(Markdown(f"#### {variable}"))
    display(df_result)

#### age

Characteristic,Centralized,Federated,H1,H2,H3,H4,H5
mean (sd),31.08 (19.44),31.08 (19.44),25.36 (17.06),33.6 (19.53),29.86 (19.96),21.54 (14.08),29.68 (18.79)
median,25.0,25.42,19.0,29.0,22.0,17.0,24.0
q1 - q3,15.0 - 43.0,15.91 - 43.07,14.0 - 31.0,17.0 - 48.0,15.0 - 41.25,13.25 - 24.0,15.0 - 40.0
min-max,7 - 91,7 - 91,7 - 89,7 - 91,7 - 91,7 - 80,7 - 90
missing,0.0,0.0,0.0,0.0,0.0,0.0,0.0
n_data,10284.0,10284.0,1233.0,5543.0,2464.0,294.0,750.0
0-9,440.0 (4.28%),440.0 (4.28%),70.0 (5.68%),190.0 (3.43%),125.0 (5.07%),18.0 (6.12%),37.0 (4.93%)
10-19,3541.0 (34.43%),3541.0 (34.43%),590.0 (47.85%),1578.0 (28.47%),949.0 (38.51%),155.0 (52.72%),269.0 (35.87%)
20-29,1977.0 (19.22%),1977.0 (19.22%),237.0 (19.22%),1042.0 (18.8%),484.0 (19.64%),75.0 (25.51%),139.0 (18.53%)
30-39,1334.0 (12.97%),1334.0 (12.97%),120.0 (9.73%),826.0 (14.9%),255.0 (10.35%),21.0 (7.14%),112.0 (14.93%)


#### sex

Characteristic,Centralized,Federated,H1,H2,H3,H4,H5
missing,0.0,0.0,0.0,0.0,0.0,0.0,0.0
n_data,10284.0,10284.0,1233.0,5543.0,2464.0,294.0,750.0
Male,5117 (49.76%),5117.0 (49.76%),680 (55.15%),2683 (48.4%),1232 (50.0%),112 (38.1%),410 (54.67%)
Female,5167 (50.24%),5167.0 (50.24%),553 (44.85%),2860 (51.6%),1232 (50.0%),182 (61.9%),340 (45.33%)


#### tissue_sample_taken

...

Characteristic,Centralized,Federated,H1,H2,H3,H4,H5
missing,1985.0,1985.0,367.0,1095.0,251.0,95.0,177.0
n_data,8299.0,8299.0,866.0,4448.0,2213.0,199.0,573.0
No,8128 (97.94%),8128.0 (97.94%),849 (98.04%),4343 (97.64%),2176 (98.33%),190 (95.48%),570 (99.48%)
Yes,171 (2.06%),171.0 (2.06%),17 (1.96%),105 (2.36%),37 (1.67%),9 (4.52%),3 (0.52%)


#### severe_bleeding

Characteristic,Centralized,Federated,H1,H2,H3,H4,H5
missing,1985.0,1985.0,367.0,1095.0,251.0,95.0,177.0
n_data,8299.0,8299.0,866.0,4448.0,2213.0,199.0,573.0
No,8184 (98.61%),8184.0 (98.61%),820 (94.69%),4419 (99.35%),2197 (99.28%),184 (92.46%),564 (98.43%)
Yes,115 (1.39%),115.0 (1.39%),46 (5.31%),29 (0.65%),16 (0.72%),15 (7.54%),9 (1.57%)


#### organ_damage

Characteristic,Centralized,Federated,H1,H2,H3,H4,H5
missing,1985.0,1985.0,367.0,1095.0,251.0,95.0,177.0
n_data,8299.0,8299.0,866.0,4448.0,2213.0,199.0,573.0
No,8069 (97.23%),8069.0 (97.23%),833 (96.19%),4326 (97.26%),2169 (98.01%),183 (91.96%),558 (97.38%)
Yes,230 (2.77%),230.0 (2.77%),33 (3.81%),122 (2.74%),44 (1.99%),16 (8.04%),15 (2.62%)


## 7. Conclusions

- Institutions never share individual records, only already-aggregated local results, demonstrated directly by `local_results`: each site's local table is the only thing that ever "leaves" that site in this simulation.

- For categorical variables, the federated approach reproduces the centralized results exactly, since they only require summing counts across sites.

- For continuous variables, most metrics (mean, standard deviation, min, max, missing values, records with a value) are also exact, since they derive from additive quantities (sums, counts) that can be recombined without the raw data. The exception is the median and quartiles: as order statistics, they can only be approximated in a federated setting, and will generally differ from their exact centralized value.

- Reproducing even the "exact" metrics federated requires more than just each site's final local statistic: `sd` needed each site to also share its sum and sum of squares, intermediate values not normally shown as descriptive results on their own. Not every locally computed value is enough on its own; the right combinable summaries need to be defined in advance.

- Because the aggregation is weighted by each site's record count, institutions with more data (e.g. H2, with 5,543 records) have proportionally more influence on the combined result than smaller ones (e.g. H4, with 294 records).

- The gap between the federated approximation and the exact centralized value for median/quartiles was small in this dataset (see section 6), but that gap depends on how differently each site's data is distributed: the more heterogeneous the sites, the larger the approximation error would be expected to be.